In [1]:
import os, time, json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import ollama

CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
EMBED_MODEL   = "all-MiniLM-L6-v2"
LLM_MODEL     = "mistral:7b-instruct"
TOP_K         = 3

embedder = SentenceTransformer(EMBED_MODEL)
print("Ready!")

/Users/sreesahithigundapaneni/ids568-milestone6-sgund12/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sreesahithigundapaneni/ids568-milestone6-sgund12/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready!


In [2]:
DOCUMENTS = [
    {"id": "doc1", "title": "RAG Overview",
     "text": """Retrieval-Augmented Generation (RAG) combines a retriever and a generator.
     The retriever finds relevant documents from a vector store using semantic similarity.
     The generator uses the retrieved context to produce grounded answers.
     RAG reduces hallucinations because the model references real documents.
     Key components are: chunker, embedder, vector index, retriever, and generator."""},

    {"id": "doc2", "title": "Vector Databases",
     "text": """FAISS is an open-source library for efficient similarity search over dense vectors.
     It supports exact and approximate nearest-neighbor search.
     Chroma is another open-source vector database that adds metadata filtering.
     Both are widely used in RAG pipelines.
     FAISS is faster for pure retrieval; Chroma is easier for filtered search."""},

    {"id": "doc3", "title": "Chunking Strategies",
     "text": """Fixed-size chunking splits documents into equal-length segments with overlap.
     Semantic chunking splits on sentence or paragraph boundaries.
     Smaller chunks give more precise retrieval but lose context.
     Larger chunks keep more context but may retrieve irrelevant content.
     A chunk size of 512 with 10 percent overlap is a common starting point."""},

    {"id": "doc4", "title": "Embedding Models",
     "text": """Sentence transformers convert text into dense vector representations.
     all-MiniLM-L6-v2 produces 384-dimensional embeddings and is fast and lightweight.
     Embeddings capture semantic meaning, enabling similarity search beyond keyword matching.
     Cosine similarity is used to rank retrieved documents by relevance."""},

    {"id": "doc5", "title": "Agent Controllers",
     "text": """An agent controller orchestrates multiple tools to complete multi-step tasks.
     The controller receives a task, selects appropriate tools, and uses results to plan next steps.
     Tool selection can be rule-based or LLM-driven.
     Common tools include retrievers, summarizers, classifiers, and calculators.
     Transparency in tool selection is critical for debugging and trust."""},

    {"id": "doc6", "title": "Evaluation Metrics",
     "text": """Retrieval evaluation uses Precision at K and Recall at K metrics.
     Precision at K measures the fraction of top-K results that are relevant.
     Recall at K measures the fraction of relevant documents retrieved in top-K.
     Hallucination rate counts statements not supported by retrieved documents.
     End-to-end latency equals retrieval latency plus generation latency."""},

    {"id": "doc7", "title": "LLM Serving Methods",
     "text": """Ollama is the easiest way to run open-weight LLMs locally on a Mac.
     It supports Mistral, Llama, Qwen, and many other models.
     vLLM is a high-throughput serving framework for cloud GPUs.
     Hugging Face Transformers is the most flexible option but needs more setup.
     For local development, Ollama with Mistral-7B is the recommended setup."""},

    {"id": "doc8", "title": "MLOps and RAG in Production",
     "text": """Production RAG systems require monitoring of retrieval quality and latency.
     Drift in document distribution can degrade retrieval performance over time.
     Reranking models improve precision by re-scoring retrieved chunks.
     Hybrid search combines dense retrieval with BM25 sparse retrieval for better recall.
     Logging every retrieval and generation step is essential for debugging."""},
]

print(f"Loaded {len(DOCUMENTS)} documents")

Loaded 8 documents


In [3]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks

all_chunks, chunk_metadata = [], []

for doc in DOCUMENTS:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append(chunk)
        chunk_metadata.append({"doc_id": doc["id"], "title": doc["title"], "chunk_index": i})

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 8


In [4]:
print("Generating embeddings...")
t0 = time.time()
embeddings = embedder.encode(all_chunks, show_progress_bar=True, convert_to_numpy=True)
print(f"Done in {time.time()-t0:.1f}s | Shape: {embeddings.shape}")

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS index ready — {index.ntotal} vectors")

Generating embeddings...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.90s/it]

Done in 1.9s | Shape: (8, 384)
FAISS index ready — 8 vectors


In [5]:
def retrieve(query, top_k=TOP_K):
    t0 = time.time()
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, top_k)
    latency_ms = (time.time() - t0) * 1000
    return [
        {"chunk": all_chunks[idx], "metadata": chunk_metadata[idx],
         "score": float(score), "retrieval_latency_ms": latency_ms}
        for score, idx in zip(scores[0], indices[0])
    ]

# Test it
for r in retrieve("What is FAISS?"):
    print(f"  {r['score']:.3f} | {r['metadata']['title']}")

  0.444 | Vector Databases
  0.033 | Embedding Models
  0.032 | Evaluation Metrics


In [6]:
def generate(query, context_chunks):
    context = "\n\n".join([
        f"[Source: {c['metadata']['title']}]\n{c['chunk']}"
        for c in context_chunks
    ])
    prompt = f"""Use ONLY the context below to answer. If the answer isn't there, say so.

Context:
{context}

Question: {query}
Answer:"""

    t0 = time.time()
    response = ollama.chat(model=LLM_MODEL,
                           messages=[{"role": "user", "content": prompt}])
    return {
        "answer": response["message"]["content"],
        "generation_latency_s": time.time() - t0
    }

# Test — takes about 10-15 seconds
test = generate("What is FAISS?", retrieve("What is FAISS?"))
print(test["answer"])
print(f"\nLatency: {test['generation_latency_s']:.1f}s")

 FAISS is an open-source library for efficient similarity search over dense vectors, and it supports both exact and approximate nearest-neighbor search. It is widely used in RAG pipelines.

Latency: 28.5s


In [7]:
def rag_pipeline(query):
    t0 = time.time()
    retrieved = retrieve(query)
    gen = generate(query, retrieved)
    return {
        "query": query,
        "retrieved_chunks": retrieved,
        "answer": gen["answer"],
        "latency": {
            "retrieval_ms": retrieved[0]["retrieval_latency_ms"],
            "generation_s": gen["generation_latency_s"],
            "total_s": time.time() - t0
        }
    }

# Quick test
r = rag_pipeline("How does chunking affect retrieval quality?")
print(r["answer"])
print(f"\nTotal latency: {r['latency']['total_s']:.1f}s")

 Chunking affects retrieval quality by finding a balance between precision and context retention. Smaller chunks offer more precise retrieval but may lead to loss of context, while larger chunks maintain more context but might retrieve irrelevant content. A common starting point is a chunk size of 512 with 10 percent overlap.

Total latency: 15.4s


In [8]:
EVAL_QUERIES = [
    {"query": "What is Retrieval-Augmented Generation?",       "relevant_docs": ["doc1"]},
    {"query": "How does FAISS differ from Chroma?",            "relevant_docs": ["doc2"]},
    {"query": "What chunk size is recommended for RAG?",       "relevant_docs": ["doc3"]},
    {"query": "What does all-MiniLM-L6-v2 produce?",          "relevant_docs": ["doc4"]},
    {"query": "How does an agent controller select tools?",    "relevant_docs": ["doc5"]},
    {"query": "What is Precision at K?",                       "relevant_docs": ["doc6"]},
    {"query": "How do I serve Mistral 7B locally?",            "relevant_docs": ["doc7"]},
    {"query": "Why is monitoring important in RAG systems?",   "relevant_docs": ["doc8"]},
    {"query": "What causes hallucination in language models?", "relevant_docs": ["doc1","doc6"]},
    {"query": "What is BM25 hybrid search?",                   "relevant_docs": ["doc8"]},
]

def precision_at_k(retrieved, relevant, k=TOP_K):
    ids = [r["metadata"]["doc_id"] for r in retrieved[:k]]
    return sum(1 for d in ids if d in relevant) / k

def recall_at_k(retrieved, relevant, k=TOP_K):
    ids = [r["metadata"]["doc_id"] for r in retrieved[:k]]
    return sum(1 for d in relevant if d in ids) / len(relevant)

eval_results = []
print(f"{'#':<3} {'Query':<48} {'P@3':>4} {'R@3':>4} {'Ret(ms)':>8} {'Gen(s)':>6}")
print("-" * 75)

for i, eq in enumerate(EVAL_QUERIES, 1):
    print(f"Running query {i}/10: {eq['query'][:40]}...")
    result = rag_pipeline(eq["query"])
    p = precision_at_k(result["retrieved_chunks"], eq["relevant_docs"])
    r = recall_at_k(result["retrieved_chunks"], eq["relevant_docs"])
    eval_results.append({**result, "precision_at_3": p, "recall_at_3": r,
                         "relevant_docs": eq["relevant_docs"]})
    print(f"{i:<3} {eq['query'][:47]:<48} {p:>4.2f} {r:>4.2f} "
          f"{result['latency']['retrieval_ms']:>8.1f} {result['latency']['generation_s']:>6.1f}")

avg_p = np.mean([r["precision_at_3"] for r in eval_results])
avg_r = np.mean([r["recall_at_3"]    for r in eval_results])
print("-" * 75)
print(f"{'AVG':<52} {avg_p:>4.2f} {avg_r:>4.2f}")

with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("\nSaved eval_results.json ✓")

#   Query                                             P@3  R@3  Ret(ms) Gen(s)
---------------------------------------------------------------------------
Running query 1/10: What is Retrieval-Augmented Generation?...
1   What is Retrieval-Augmented Generation?          0.33 1.00   2417.9   13.1
Running query 2/10: How does FAISS differ from Chroma?...
2   How does FAISS differ from Chroma?               0.33 1.00   2372.3   15.1
Running query 3/10: What chunk size is recommended for RAG?...
3   What chunk size is recommended for RAG?          0.33 1.00    560.1   14.0
Running query 4/10: What does all-MiniLM-L6-v2 produce?...
4   What does all-MiniLM-L6-v2 produce?              0.33 1.00   1094.3    7.8
Running query 5/10: How does an agent controller select tool...
5   How does an agent controller select tools?       0.33 1.00    359.5   14.3
Running query 6/10: What is Precision at K?...
6   What is Precision at K?                          0.33 1.00   1722.5    7.2
Running query 7/1

In [9]:
# ── Latency Summary ───────────────────────────────────────────────
import json

with open("eval_results.json") as f:
    results = json.load(f)

ret_latencies   = [r["latency"]["retrieval_ms"] for r in results]
gen_latencies   = [r["latency"]["generation_s"] for r in results]
total_latencies = [r["latency"]["total_s"]       for r in results]

print("=" * 55)
print("LATENCY MEASUREMENTS (10 evaluation queries)")
print("=" * 55)
print(f"Retrieval  — min: {min(ret_latencies):.0f}ms  max: {max(ret_latencies):.0f}ms  avg: {sum(ret_latencies)/len(ret_latencies):.0f}ms")
print(f"Generation — min: {min(gen_latencies):.1f}s   max: {max(gen_latencies):.1f}s   avg: {sum(gen_latencies)/len(gen_latencies):.1f}s")
print(f"End-to-end — min: {min(total_latencies):.1f}s   max: {max(total_latencies):.1f}s   avg: {sum(total_latencies)/len(total_latencies):.1f}s")
print("=" * 55)
print(f"Model: mistral:7b-instruct (7B) via Ollama on Mac CPU")

LATENCY MEASUREMENTS (10 evaluation queries)
Retrieval  — min: 325ms  max: 2418ms  avg: 1225ms
Generation — min: 7.2s   max: 15.4s   avg: 12.6s
End-to-end — min: 8.9s   max: 17.5s   avg: 13.8s
Model: mistral:7b-instruct (7B) via Ollama on Mac CPU
